# Bank Customer Churn - EDA & Modeling

This notebook explores the Bank Customer Churn dataset and builds predictive models.

## 1. Load Libraries & Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Load data
df = pd.read_csv('../data/bank_churn.csv')
print("Dataset shape:", df.shape)
df.head()

## 2. Data Overview

In [ ]:
print("\n=== Dataset Info ===")
print(df.info())

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Summary Statistics ===")
print(df.describe())

## 3. Target Variable Analysis

In [ ]:
# Target distribution
print("Churn Distribution:")
print(df['Exited'].value_counts())
print(f"\nChurn Rate: {df['Exited'].mean():.1%}")

# Plot
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df['Exited'].value_counts().plot(kind='bar', ax=ax[0])
ax[0].set_title('Churn Count')
ax[0].set_xlabel('Exited (0=No, 1=Yes)')

df['Exited'].value_counts(normalize=True).plot(kind='pie', ax=ax[1], autopct='%1.1f%%')
ax[1].set_title('Churn Distribution')
ax[1].set_ylabel('')
plt.tight_layout()
plt.show()

## 4. Numeric Features Analysis

In [ ]:
# Numeric columns
numeric_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'EstimatedSalary']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    axes[idx].hist(df[col], bins=30, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{col} Distribution')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Count')

# Remove extra subplot
axes[-1].remove()
plt.tight_layout()
plt.show()

print("\nNumeric Features Summary:")
print(df[numeric_cols].describe())

## 5. Categorical Features Analysis

In [ ]:
# Categorical columns
categorical_cols = ['Geography', 'Gender', 'NumOfProducts']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, col in enumerate(categorical_cols):
    df[col].value_counts().plot(kind='bar', ax=axes[idx])
    axes[idx].set_title(f'{col} Distribution')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Count')
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 6. Churn by Key Features

In [ ]:
# Churn by Geography
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# By Geography
churn_by_geo = df.groupby('Geography')['Exited'].mean()
churn_by_geo.plot(kind='bar', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Churn Rate by Geography', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Churn Rate')
axes[0, 0].tick_params(axis='x', rotation=45)

# By Gender
churn_by_gender = df.groupby('Gender')['Exited'].mean()
churn_by_gender.plot(kind='bar', ax=axes[0, 1], color='salmon')
axes[0, 1].set_title('Churn Rate by Gender', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Churn Rate')
axes[0, 1].tick_params(axis='x', rotation=45)

# By NumOfProducts
churn_by_products = df.groupby('NumOfProducts')['Exited'].mean()
churn_by_products.plot(kind='bar', ax=axes[1, 0], color='lightgreen')
axes[1, 0].set_title('Churn Rate by Number of Products', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Churn Rate')
axes[1, 0].set_xlabel('NumOfProducts')

# Age box plot
df.boxplot(column='Age', by='Exited', ax=axes[1, 1])
axes[1, 1].set_title('Age Distribution by Churn Status', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Age')
axes[1, 1].set_xlabel('Exited (0=No, 1=Yes)')
plt.suptitle('')

plt.tight_layout()
plt.show()

print("\nChurn Rates by Key Features:")
print(f"By Geography:\n{churn_by_geo}\n")
print(f"By Gender:\n{churn_by_gender}\n")
print(f"By NumOfProducts:\n{churn_by_products}")

## 7. Correlation Analysis

In [ ]:
# Create a copy for correlation analysis
df_corr = df.copy()
df_corr['Geography_Germany'] = (df_corr['Geography'] == 'Germany').astype(int)
df_corr['Gender_Male'] = (df_corr['Gender'] == 'Male').astype(int)

# Select numeric and encoded columns
corr_cols = ['Exited', 'CreditScore', 'Age', 'Tenure', 'Balance', 'EstimatedSalary', 
             'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'Geography_Germany', 'Gender_Male']

correlation = df_corr[corr_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            cbar_kws={'label': 'Correlation'}, square=True)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop Features Correlated with Churn:")
churn_corr = correlation['Exited'].sort_values(ascending=False)
print(churn_corr)

## 8. Key Insights

### Main Findings from EDA:

1. **Class Imbalance**: Churn rate is ~31.3% - slightly imbalanced but manageable with SMOTE.

2. **Geography Effect**: Customers in Germany have significantly higher churn rate (~32.4%) compared to France (~16.2%) and Spain (~16.8%). This is the strongest geographic signal.

3. **Age Impact**: Older customers (40+) show much higher churn rates. Age is one of the top predictors of churn.

4. **Product Count**: Customers with more products are less likely to churn. Having 1 product shows ~27.7% churn vs 1 product = ~27.7% and 4 products = ~10.1%.

5. **Active Membership**: Active members have significantly lower churn rates (~25.5%) vs inactive (~37.6%). This suggests engagement is protective.

6. **Credit Score**: Generally weak correlation with churn, but very low credit scores (<500) show slightly elevated churn.

7. **Balance**: Zero balance customers show higher churn propensity. This could indicate dissatisfaction or dormancy.

8. **Gender**: Females have slightly higher churn rate (~25.1%) vs Males (~23.5%) - a subtle but consistent effect.

### Implications for Modeling:
- Focus feature engineering on Age, Geography, IsActiveMember, NumOfProducts, and Balance
- Use SMOTE or class weights to handle imbalance
- Tree-based models (XGBoost, LightGBM) should capture non-linear age effects and interactions well

## 9. Quick Model Comparison

In [ ]:
# Quick preprocessing for initial model comparison
df_model = df.copy()

# Drop unnecessary columns
df_model = df_model.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

# One-hot encode
df_model = pd.get_dummies(df_model, columns=['Geography', 'Gender'], drop_first=False)

X = df_model.drop('Exited', axis=1)
y = df_model['Exited']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Churn rate in train: {y_train.mean():.1%}")
print(f"Churn rate in test: {y_test.mean():.1%}")

In [ ]:
# Train quick models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
}

results = {}

for name, model in models.items():
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'AUC_ROC': roc_auc_score(y_test, y_pred_proba)
    }

results_df = pd.DataFrame(results).T
print("\nModel Comparison:")
print(results_df.round(4))